# IFRS 9 Expected Credit Loss (ECL) Model
## Notebook 5 — Stress Testing & Sensitivity Analysis

This notebook quantifies how ECL responds to changes in model parameters,
supporting risk management decisions and regulatory reporting.

**Analyses:**
1. PD sensitivity (±50%, +100%)
2. LGD sensitivity (±20%)
3. Combined stress scenarios
4. Stage migration analysis
5. Sector concentration risk

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import generate_loan_portfolio, assign_stage
from src.pd_model import PDModel
from src.lgd_ead import compute_ead, compute_lgd
from src.ecl_calculator import ECLCalculator

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_style('whitegrid')

df = assign_stage(generate_loan_portfolio(n_loans=5000, random_state=42))
pd_model = PDModel(macro_scalar=1.0)
df = pd_model.assign_pd(df)
df['ead'] = compute_ead(df)
df['lgd'] = compute_lgd(df)
calc = ECLCalculator(discount_rate=0.05)
df = calc.compute_ecl(df)
base_ecl = df['ecl'].sum()

print(f'Base ECL: THB {base_ecl/1e6:.1f}M  |  Coverage: {base_ecl/df["ead"].sum():.2%}')

## 1. PD & LGD Sensitivity Analysis

In [ ]:
sens = calc.sensitivity_analysis(df)

print('=== PD Sensitivity ===')
print(sens['pd_sensitivity'].to_string(index=False))

print('\n=== LGD Sensitivity ===')
print(sens['lgd_sensitivity'].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pd_sens = sens['pd_sensitivity']
bar_colors = ['#27ae60' if v <= 0 else '#e74c3c' for v in pd_sens['pct_change']]
axes[0].bar(pd_sens['pd_shock'], pd_sens['pct_change'], color=bar_colors, alpha=0.85)
axes[0].axhline(0, color='black', lw=0.8)
axes[0].set_title('ECL Change vs PD Shock')
axes[0].set_xlabel('PD Shock')
axes[0].set_ylabel('ECL Change (%)')

lgd_sens = sens['lgd_sensitivity']
bar_colors2 = ['#27ae60' if v <= 0 else '#e74c3c' for v in lgd_sens['pct_change']]
axes[1].bar(lgd_sens['lgd_shock'], lgd_sens['pct_change'], color=bar_colors2, alpha=0.85)
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('ECL Change vs LGD Shock')
axes[1].set_xlabel('LGD Shock')
axes[1].set_ylabel('ECL Change (%)')

plt.suptitle('ECL Sensitivity Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/05_sensitivity.png', bbox_inches='tight')
plt.show()

## 2. Combined Stress Scenarios

In [ ]:
stress_scenarios = [
    {'name': 'Base', 'pd_shock': 0.0, 'lgd_shock': 0.0},
    {'name': 'Mild Stress', 'pd_shock': 0.25, 'lgd_shock': 0.10},
    {'name': 'Moderate Stress', 'pd_shock': 0.50, 'lgd_shock': 0.20},
    {'name': 'Severe Stress', 'pd_shock': 1.00, 'lgd_shock': 0.30},
    {'name': 'Extreme Stress', 'pd_shock': 2.00, 'lgd_shock': 0.50},
]

stress_results = []
for sc in stress_scenarios:
    stressed_pd = (df['pd_applied'] * (1 + sc['pd_shock'])).clip(0, 1)
    stressed_lgd = (df['lgd'] * (1 + sc['lgd_shock'])).clip(0, 1)
    stressed_ecl = (stressed_pd * stressed_lgd * df['ead']).sum()
    stress_results.append({
        'Scenario': sc['name'],
        'PD Shock': f"+{sc['pd_shock']:.0%}",
        'LGD Shock': f"+{sc['lgd_shock']:.0%}",
        'ECL (THB M)': round(stressed_ecl / 1e6, 1),
        'vs Base (%)': round((stressed_ecl - base_ecl) / base_ecl * 100, 1),
        'Coverage (%)': round(stressed_ecl / df['ead'].sum() * 100, 2),
    })

stress_df = pd.DataFrame(stress_results)
print('=== Combined Stress Test Results ===')
print(stress_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colors_stress = ['#27ae60','#f1c40f','#e67e22','#e74c3c','#8e44ad']
bars = ax.bar(stress_df['Scenario'], stress_df['ECL (THB M)'], color=colors_stress, alpha=0.85)
ax.axhline(base_ecl / 1e6, color='black', linestyle='--', lw=1.5, label='Base ECL')
for bar, val in zip(bars, stress_df['ECL (THB M)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{val:.0f}M', ha='center', fontsize=9)
ax.set_title('ECL Under Combined Stress Scenarios', fontweight='bold')
ax.set_ylabel('ECL (THB Million)')
ax.legend()
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig('../plots/05_stress_scenarios.png', bbox_inches='tight')
plt.show()

## 3. Stage Migration Simulation

In [ ]:
# Simulate stage migration: 10% of Stage 1 → Stage 2, 5% of Stage 2 → Stage 3
np.random.seed(42)

df_migrated = df.copy()
stage1_idx = df_migrated[df_migrated['stage'] == 1].index
stage2_idx = df_migrated[df_migrated['stage'] == 2].index

migrate_1_to_2 = np.random.choice(stage1_idx, size=int(len(stage1_idx) * 0.10), replace=False)
migrate_2_to_3 = np.random.choice(stage2_idx, size=int(len(stage2_idx) * 0.05), replace=False)

df_migrated.loc[migrate_1_to_2, 'stage'] = 2
df_migrated.loc[migrate_2_to_3, 'stage'] = 3
df_migrated.loc[df_migrated['stage'].isin([2, 3]), 'pd_applied'] = \
    df_migrated.loc[df_migrated['stage'].isin([2, 3]), 'pd_lifetime']

df_migrated = calc.compute_ecl(df_migrated)
migrated_ecl = df_migrated['ecl'].sum()

print('=== Stage Migration Impact ===')
print(f'Base ECL:     THB {base_ecl/1e6:.1f}M')
print(f'Migrated ECL: THB {migrated_ecl/1e6:.1f}M')
print(f'ECL Increase: THB {(migrated_ecl-base_ecl)/1e6:.1f}M ({(migrated_ecl-base_ecl)/base_ecl:.1%})')

stage_comparison = pd.DataFrame({
    'Before Migration': df.groupby('stage')['ecl'].sum() / 1e6,
    'After Migration':  df_migrated.groupby('stage')['ecl'].sum() / 1e6,
})
print('\nECL by Stage (THB M):')
print(stage_comparison.round(1))